<a href="https://colab.research.google.com/github/vaidysrm-gitub/TMDB-Analystics/blob/main/2_TMDB_SQL_Data_Loading%26Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Install the required Python packages
!pip install mysql-connector-python pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 37.4 MB/s  0:00:00


In [2]:
#Upload your six CSV files(Cleaning files) into Google Colab
from google.colab import files

uploaded = files.upload()

Saving movie_genres.csv to movie_genres.csv
Saving movies.csv to movies.csv
Saving movies_keywords.csv to movies_keywords.csv
Saving cast.csv to cast.csv
Saving crew.csv to crew.csv
Saving genres.csv to genres.csv


In [3]:
# Import the Python libraries
import pandas as pd
import mysql.connector
from datetime import date

In [4]:
#Upload your CA certificate into Colab using:
from google.colab import files

uploaded = files.upload()

Saving isrgrootx1.pem to isrgrootx1.pem


# **#Connection MYSQL**

In [5]:
connection=mysql.connector.connect(host="gateway01.ap-southeast-1.prod.aws.tidbcloud.com"
                                   ,user="2iciJQar3qvYtst.root",password="t0v0FXoYwC8OsiH5", port=4000, ssl_ca="/content/isrgrootx1.pem",
    ssl_verify_cert=True)
print("Database connection successful!")

Database connection successful!


In [6]:
#Create your movie_database

cursor = connection.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS movie_database")

cursor.execute("USE movie_database")

print("movie_database selected successfully!")

movie_database selected successfully!


In [7]:
# TABLE 1: MOVIES

cursor.execute("""
CREATE TABLE IF NOT EXISTS movies (
    movie_id INT PRIMARY KEY,
    title VARCHAR(255),
    original_language VARCHAR(10),
    release_date DATE,
    runtime INT,
    budget BIGINT,
    revenue BIGINT,
    popularity FLOAT,
    vote_average FLOAT,
    vote_count INT,
    status VARCHAR(50),
    overview TEXT
)
""")

# TABLE 2: GENRES

cursor.execute("""
CREATE TABLE IF NOT EXISTS genres (
    genre_id INT PRIMARY KEY,
    genre_name VARCHAR(50)
)
""")

# TABLE 3: MOVIE_GENRES

cursor.execute("""
CREATE TABLE IF NOT EXISTS movie_genres (
    movie_id INT,
    genre_id INT,
    FOREIGN KEY (movie_id)
        REFERENCES movies(movie_id),
    FOREIGN KEY (genre_id)
        REFERENCES genres(genre_id)
)
""")

# TABLE 4: CAST

cursor.execute("""
CREATE TABLE IF NOT EXISTS `cast` (
    cast_row_id INT PRIMARY KEY AUTO_INCREMENT,
    movie_id INT,
    person_id INT,
    actor_name VARCHAR(255),
    character_name TEXT,
    cast_order INT,
    FOREIGN KEY (movie_id)
        REFERENCES movies(movie_id)
)
""")

# TABLE 5: CREW

cursor.execute("""
CREATE TABLE IF NOT EXISTS crew (
    crew_row_id INT PRIMARY KEY AUTO_INCREMENT,
    movie_id INT,
    person_id INT,
    person_name VARCHAR(255),
    job VARCHAR(100),
    department VARCHAR(100),
    FOREIGN KEY (movie_id)
        REFERENCES movies(movie_id)
)
""")

# TABLE 6: MOVIE_KEYWORDS

cursor.execute("""
CREATE TABLE IF NOT EXISTS movie_keywords (
    movie_id INT,
    keyword_id INT,
    keyword_name VARCHAR(100),
    FOREIGN KEY (movie_id)
        REFERENCES movies(movie_id)
)
""")

connection.commit()

print("All six tables created successfully!")

All six tables created successfully!


In [8]:
cursor.execute("SHOW TABLES")

tables = cursor.fetchall()

for table in tables:
    print(table)

('cast',)
('crew',)
('genres',)
('movie_genres',)
('movie_keywords',)
('movies',)


In [9]:
#Load the CSV data into Pandas DataFrames
df_movies = pd.read_csv("movies.csv")

df_genres = pd.read_csv("genres.csv")

df_movie_genres = pd.read_csv("movie_genres.csv")

df_cast = pd.read_csv("cast.csv")

df_crew = pd.read_csv("crew.csv")

df_movie_keywords = pd.read_csv("movies_keywords.csv")

In [10]:
#Check whether all six DataFrames have loaded
print("Movies:", len(df_movies))
print("Genres:", len(df_genres))
print("Movie Genres:", len(df_movie_genres))
print("Cast:", len(df_cast))
print("Crew:", len(df_crew))
print("Movie Keywords:", len(df_movie_keywords))

Movies: 2503
Genres: 19
Movie Genres: 6878
Cast: 24902
Crew: 7289
Movie Keywords: 41681


In [11]:
df_movies.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2503 entries, 0 to 2502
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   movie_id           2503 non-null   int64  
 1   title              2503 non-null   object 
 2   original_language  2503 non-null   object 
 3   release_date       2503 non-null   object 
 4   runtime            2503 non-null   int64  
 5   budget             2503 non-null   float64
 6   revenue            2503 non-null   float64
 7   popularity         2503 non-null   float64
 8   vote_average       2503 non-null   float64
 9   vote_count         2503 non-null   int64  
 10  status             2503 non-null   object 
 11  overview           2503 non-null   object 
dtypes: float64(4), int64(3), object(5)
memory usage: 234.8+ KB


In [12]:
df_movies["release_date"] = pd.to_datetime(
    df_movies["release_date"]
).dt.date

df_movies["budget"] = df_movies["budget"].round().astype("int64")

df_movies["revenue"] = df_movies["revenue"].round().astype("int64")

print(df_movies.dtypes)

movie_id               int64
title                 object
original_language     object
release_date          object
runtime                int64
budget                 int64
revenue                int64
popularity           float64
vote_average         float64
vote_count             int64
status                object
overview              object
dtype: object


In [13]:
print(df_movies["release_date"].head(10))

print("\nMissing release dates:")

print(df_movies["release_date"].isnull().sum())

0    2014-11-05
1    2010-07-15
2    2012-04-25
3    2008-07-16
4    2009-12-16
5    2016-02-09
6    1999-10-15
7    2018-04-25
8    1994-09-23
9    1994-09-10
Name: release_date, dtype: object

Missing release dates:
0


In [14]:
cursor.execute("USE movie_database")

cursor.execute("SELECT COUNT(*) FROM movies")

print(cursor.fetchone())

(2503,)


In [15]:
print("Total movie records:", len(df_movies))

print(df_movies.head(5))

Total movie records: 2503
   movie_id            title original_language release_date  runtime  \
0    157336     Interstellar                en   2014-11-05      169   
1     27205        Inception                en   2010-07-15      148   
2     24428     The Avengers                en   2012-04-25      143   
3       155  The Dark Knight                en   2008-07-16      152   
4     19995           Avatar                en   2009-12-16      162   

      budget     revenue  popularity  vote_average  vote_count    status  \
0  165000000   746606706     73.0452         8.484       40566  Released   
1  160000000   839030630     55.8497         8.400       39727  Released   
2  220000000  1518815515     72.2515         8.062       38993  Released   
3  185000000  1004558444     60.5510         8.534       36255  Released   
4  237000000  2923706026     48.1996         7.610       34348  Released   

                                            overview  
0  The adventures of a group 

In [ ]:
#our TMDB project contains six tables movies, genres, movies_genres, cast, crew and movie_keyword in movie_database
# we already load the csv files into  Pandas DataFrames and assigned name as (df_movies, df_genres, df_movie_genres, df_cast, df_crew and df_movie_keywords)
# Now we upload movies table
# Convert DataFrame into a list
values = df_movies.astype(object).where(
    pd.notna(df_movies), None
).values.tolist()
#insert statement(six tables)
insert = """
INSERT INTO movies
(movie_id, title, original_language, release_date,
runtime, budget, revenue, popularity, vote_average,
vote_count, status, overview)

VALUES (%s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s, %s)
"""

In [ ]:
#insert records
cursor.executemany(insert, values)
connection.commit()

NameError: name 'insert' is not defined

In [ ]:
print("Movies uploaded successfully!")

Movies uploaded successfully!


In [ ]:
values = df_genres.values.tolist()

insert = """
INSERT INTO genres
(genre_id, genre_name)
VALUES (%s, %s)
"""

cursor.executemany(insert, values)

connection.commit()

print("Genres uploaded successfully!")

ProgrammingError: Failed processing format-parameters; MySQL Connection not available

In [ ]:
values = df_movie_genres.values.tolist()

insert = """
INSERT INTO movie_genres
(movie_id, genre_id)
VALUES (%s, %s)
"""

cursor.executemany(insert, values)

connection.commit()

print("Movie Genres uploaded successfully!")

Movie Genres uploaded successfully!


In [ ]:
values = df_cast.astype(object).where(
    pd.notna(df_cast), None
).values.tolist()

insert = """
INSERT INTO `cast`
(cast_row_id, movie_id, person_id,
actor_name, character_name, cast_order)

VALUES (%s, %s, %s, %s, %s, %s)
"""

cursor.executemany(insert, values)

connection.commit()

print("Cast uploaded successfully!")

Cast uploaded successfully!


In [ ]:
values = df_crew.astype(object).where(
    pd.notna(df_crew), None
).values.tolist()

insert = """
INSERT INTO crew
(crew_row_id, movie_id, person_id,
person_name, job, department)

VALUES (%s, %s, %s, %s, %s, %s)
"""

cursor.executemany(insert, values)

connection.commit()

print("Crew uploaded successfully!")

Crew uploaded successfully!


In [ ]:
values = df_movie_keywords.values.tolist()

insert = """
INSERT INTO movie_keywords
(movie_id, keyword_id, keyword_name)

VALUES (%s, %s, %s)
"""

cursor.executemany(insert, values)

connection.commit()

print("Movie Keywords uploaded successfully!")

Movie Keywords uploaded successfully!


In [16]:
tables = [
    "movies",
    "genres",
    "movie_genres",
    "cast",
    "crew",
    "movie_keywords"
]

for table in tables:

    cursor.execute(
        f"SELECT COUNT(*) FROM `{table}`"
    )

    result = cursor.fetchone()

    print(table, ":", result[0])

movies : 2503
genres : 19
movie_genres : 6878
cast : 24902
crew : 7289
movie_keywords : 41681


# **SQL Analysis — 10 Questions**

# 1. List all movies in the Action genre released after 2015, showing title and release date.

In [17]:
#movies=m, movie_genres=mg, genres=g
#m.movie_id(PK),g.genre_id(PK),mg.movie_id(FK->movies) & mg.genre_id(FK->genres)
cursor.execute("""select m.title, m.release_date
from movies m

join movie_genres mg
on m.movie_id=mg.movie_id

join genres g
on mg.genre_id=g.genre_id

where g.genre_name ='Action'and m.release_date >= '2016-01-01'

ORDER BY m.release_date DESC


;""")
rows = cursor.fetchall()

column_names = [desc[0] for desc in cursor.description]

data = pd.DataFrame(rows,columns=column_names)
data

,title,release_date
0,Mortal Kombat II,2026-05-06
1,Predator: Badlands,2025-11-05
2,The Fantastic 4: First Steps,2025-07-23
3,Superman,2025-07-09
4,F1,2025-06-25
...,...,...
263,Triple 9,2016-02-19
264,Deadpool,2016-02-09
265,Zoolander 2,2016-02-06
266,Kung Fu Panda 3,2016-01-23


# 2. Find the top 10 highest-grossing movies along with the genre(s) each belongs to.

In [18]:
#movies=m, movie_genres=mg,genres=g
#m.movie_id(PK),g.genre_id(PK),mg.movie_id(FK->movies) & mg.genre_id(FK->genres)
cursor.execute("""select m.title, m.revenue, g.genre_name
from movies m

LEFT join movie_genres mg
on m.movie_id=mg.movie_id

LEFT join genres g
on mg.genre_id=g.genre_id

GROUP BY m.title, m.revenue, g.genre_name

ORDER BY m.revenue DESC

LIMIT 10

;""")
rows = cursor.fetchall()

column_names = [desc[0] for desc in cursor.description]

data = pd.DataFrame(rows,columns=column_names)
data

,title,revenue,genre_name
0,Avatar,2923706026,Adventure
1,Avatar,2923706026,Action
2,Avatar,2923706026,Science Fiction
3,Avengers: Endgame,2799439100,Adventure
4,Avengers: Endgame,2799439100,Action
5,Avengers: Endgame,2799439100,Science Fiction
6,Avatar: The Way of Water,2334484620,Adventure
7,Avatar: The Way of Water,2334484620,Science Fiction
8,Avatar: The Way of Water,2334484620,Action
9,Titanic,2264162353,Drama


# 3. Calculate the average budget and average revenue for each genre, sorted by average revenue descending.

In [19]:
#movies=m, movie_genres=mg,genres=g
#m.movie_id(PK),g.genre_id(PK),mg.movie_id(FK->movies) & mg.genre_id(FK->genres)

cursor.execute(""" SELECT g.genre_name, ROUND(AVG(m.budget), 2) AS average_budget, ROUND(AVG(m.revenue), 2) AS average_revenue

FROM movies m

JOIN movie_genres mg
    ON m.movie_id = mg.movie_id

JOIN genres g
    ON mg.genre_id = g.genre_id

GROUP BY g.genre_name

ORDER BY average_revenue DESC
""")
rows = cursor.fetchall()

column_names = [desc[0] for desc in cursor.description]

data = pd.DataFrame(rows,columns=column_names)
data

,genre_name,average_budget,average_revenue
0,Adventure,110081821.14,395846129.22
1,Animation,86327260.31,367004591.83
2,Family,86882552.40,349587419.30
3,Science Fiction,96179393.41,330311525.75
4,Action,93896709.72,306134078.01
5,Fantasy,87867187.28,300499686.98
6,Documentary,60816431.00,223278963.00
7,Comedy,54723981.65,221606873.37
8,Music,43551691.06,202009291.43
9,Romance,38444826.02,178144073.99


## 4. List the top 10 actors who have appeared in the most movies in the dataset.(cast)

In [20]:
#cast=c,movies=m
cursor.execute(""" SELECT c.person_id, c.actor_name, COUNT(DISTINCT c.movie_id) AS total_movies

FROM cast c

GROUP BY
    c.person_id,
    c.actor_name

ORDER BY total_movies DESC

LIMIT 10
""")
rows = cursor.fetchall()

column_names = [desc[0] for desc in cursor.description]

data = pd.DataFrame(rows,columns=column_names)
data

,person_id,actor_name,total_movies
0,2231,Samuel L. Jackson,39
1,380,Robert De Niro,38
2,287,Brad Pitt,38
3,85,Johnny Depp,37
4,31,Tom Hanks,33
5,1245,Scarlett Johansson,32
6,13240,Mark Wahlberg,32
7,5293,Willem Dafoe,32
8,500,Tom Cruise,31
9,192,Morgan Freeman,31


# 5. Find all directors who have directed more than 3 movies, along with their movie count.

In [22]:
cursor.execute(""" SELECT cw.person_id,cw.person_name,COUNT(DISTINCT cw.movie_id) AS total_movies

FROM crew cw

WHERE job = 'Director'

GROUP BY
    cw.person_id,
    cw.person_name

HAVING COUNT(DISTINCT cw.movie_id) > 3

ORDER BY total_movies DESC
""")

rows = cursor.fetchall()

column_names = [desc[0] for desc in cursor.description]

data = pd.DataFrame(rows,columns=column_names)
data

,person_id,person_name,total_movies
0,488,Steven Spielberg,27
1,578,Ridley Scott,19
2,510,Tim Burton,19
3,1032,Martin Scorsese,16
4,865,Michael Bay,15
...,...,...,...
212,13848,Charlie Chaplin,4
213,9915,Guy Hamilton,4
214,12890,Pete Docter,4
215,1056121,Ryan Coogler,4


# 6.List the top 10 most frequently used keywords across all movies.

In [23]:
cursor.execute("""SELECT mk.keyword_id, mk.keyword_name, COUNT(DISTINCT mk.movie_id) AS total_movies

FROM movie_keywords mk

GROUP BY
    mk.keyword_id,
    mk.keyword_name

ORDER BY total_movies DESC

LIMIT 10
""")
rows = cursor.fetchall()

column_names = [desc[0] for desc in cursor.description]

data = pd.DataFrame(rows,columns=column_names)
data

,keyword_id,keyword_name,total_movies
0,818,based on novel or book,449
1,9663,sequel,416
2,179431,duringcreditsstinger,296
3,179430,aftercreditsstinger,242
4,9826,murder,176
5,9672,based on true story,166
6,242,new york city,161
7,3289,villain,160
8,278823,3d animation,155
9,9717,based on comic,154


# 7.Find movies where the budget was above the overall average budget but the revenue was below the overall average revenue.

In [24]:
cursor.execute (""" SELECT m.movie_id, m.title, m.budget, m.revenue

FROM movies m

WHERE m.budget > (
    SELECT AVG(m.budget)
    FROM movies m
)

AND m.revenue < (
    SELECT AVG(m.revenue)
    FROM movies m
)

ORDER BY budget DESC
""")
rows = cursor.fetchall()

column_names = [desc[0] for desc in cursor.description]

data = pd.DataFrame(rows,columns=column_names)
data

,movie_id,title,budget,revenue
0,609681,The Marvels,274800000,206136825
1,845781,Red One,250000000,185700759
2,812583,Wake Up Dead Man: A Knives Out Mystery,210000000,4000000
3,44912,Green Lantern,200000000,219851172
4,725201,The Gray Man,200000000,454023
...,...,...,...,...
283,1370,Rambo III,63000000,189015611
284,8202,Æon Flux,62000000,53321673
285,345887,The Equalizer 2,62000000,190400157
286,136400,2 Guns,61000000,131940411


# 8. List all actors who have appeared in a movie directed by a specific director (e.g., Christopher Nolan).

In [25]:
cursor.execute(""" SELECT DISTINCT c.actor_name, m.movie_id, m.title, cr.job, cr.person_name, cr.department

FROM cast c

JOIN movies m
    ON c.movie_id = m.movie_id

JOIN crew cr
    ON m.movie_id = cr.movie_id

WHERE cr.person_name = 'Christopher Nolan'
AND cr.job = 'Director'

ORDER BY m.title;

""")
rows = cursor.fetchall()

column_names = [desc[0] for desc in cursor.description]

data = pd.DataFrame(rows,columns=column_names)
data

,actor_name,movie_id,title,job,person_name,department
0,Christian Bale,272,Batman Begins,Director,Christopher Nolan,Directing
1,Cillian Murphy,272,Batman Begins,Director,Christopher Nolan,Directing
2,Rutger Hauer,272,Batman Begins,Director,Christopher Nolan,Directing
3,Mark Boone Junior,272,Batman Begins,Director,Christopher Nolan,Directing
4,Tom Wilkinson,272,Batman Begins,Director,Christopher Nolan,Directing
...,...,...,...,...,...,...
105,Piper Perabo,1124,The Prestige,Director,Christopher Nolan,Directing
106,Samantha Mahurin,1124,The Prestige,Director,Christopher Nolan,Directing
107,Scarlett Johansson,1124,The Prestige,Director,Christopher Nolan,Directing
108,Rebecca Hall,1124,The Prestige,Director,Christopher Nolan,Directing


# 9. Find the genre with the highest average rating, considering only genres with at least 20 movies.

In [26]:
cursor.execute(""" SELECT g.genre_name, ROUND(AVG(m.vote_average), 2) AS average_rating, COUNT(DISTINCT m.movie_id) AS total_movies

FROM movies m

JOIN movie_genres mg
    ON m.movie_id = mg.movie_id

JOIN genres g
    ON mg.genre_id = g.genre_id

GROUP BY g.genre_name

HAVING COUNT(DISTINCT m.movie_id) >= 20

ORDER BY average_rating DESC

""")
rows = cursor.fetchall()

column_names = [desc[0] for desc in cursor.description]

data = pd.DataFrame(rows,columns=column_names)
data


,genre_name,average_rating,total_movies
0,War,7.36,85
1,History,7.34,112
2,Music,7.32,49
3,Drama,7.25,937
4,Animation,7.20,246
5,Western,7.13,33
6,Romance,7.03,349
7,Crime,7.01,387
8,Family,6.94,344
9,Mystery,6.88,216


# 10. List the top 10 movies with the highest number of keywords tagged to them.

In [27]:
cursor.execute("""SELECT m.movie_id, m.title, COUNT(DISTINCT mk.keyword_id) AS total_keywords
FROM movies m

JOIN movie_keywords mk
    ON m.movie_id = mk.movie_id
GROUP BY
    m.movie_id,
    m.title
    ORDER BY total_keywords DESC

    LIMIT 10
    """)
rows = cursor.fetchall()

column_names = [desc[0] for desc in cursor.description]

data = pd.DataFrame(rows,columns=column_names)
data


,movie_id,title,total_keywords
0,1100782,Smile 2,98
1,260346,Taken 3,70
2,588,Silent Hill,63
3,63,Twelve Monkeys,57
4,281338,War for the Planet of the Apes,54
5,694,The Shining,52
6,119450,Dawn of the Planet of the Apes,51
7,10191,How to Train Your Dragon,50
8,166428,How to Train Your Dragon: The Hidden World,49
9,268896,Pacific Rim: Uprising,49
